In [1]:
import random
import math

In [2]:
def sigmoid(z):
    return 1.0/(1.0 + math.exp(-z))

In [5]:
random.seed(42)
N = 200
X = []
y = []

In [6]:
for _ in range(N // 2):
    X.append([random.gauss(2, 1), random.gauss(2, 1)])
    y.append(0)

for _ in range(N // 2):
    X.append([random.gauss(5, 1), random.gauss(5, 1)])
    y.append(1)

combined = list(zip(X, y))
random.shuffle(combined)
X, y = zip(*combined)
X = list(X)
y = list(y)

print(f"Generated {N} samples (2 classes, 2 features)")
print(f"Class 0 center: (2, 2), Class 1 center: (5, 5)")
print(f"First 5 samples:")
for i in range(5):
    print(f"  Features: [{X[i][0]:.2f}, {X[i][1]:.2f}], Label: {y[i]}")

Generated 200 samples (2 classes, 2 features)
Class 0 center: (2, 2), Class 1 center: (5, 5)
First 5 samples:
  Features: [5.67, 3.86], Label: 1
  Features: [0.88, 2.46], Label: 0
  Features: [1.53, 2.50], Label: 0
  Features: [4.60, 6.20], Label: 1
  Features: [6.30, 6.90], Label: 1


In [13]:
class LogisticRegression:
    def __init__(self, n_features, learning_rate=0.01):
        self.weights = [0.0] * n_features
        self.bias = 0.0
        self.lr = learning_rate
        self.loss_history = []

    def predict_proba(self, x):
        z = sum(w * xi for w, xi in zip(self.weights, x)) + self.bias
        return sigmoid(z)

    def predict(self, x, threshold=0.5):
        return 1 if self.predict_proba(x) >= threshold else 0

    def compute_loss(self, X, y):
        n = len(y)
        total = 0.0
        for i in range(n):
            p = self.predict_proba(X[i])
            p = max(1e-15, min(1 - 1e-15, p))
            total += y[i] * math.log(p) + (1 - y[i]) * math.log(1 - p)
        return -total / n

    def fit(self, X, y, epochs=1000, print_every=200):
        n = len(y)
        n_features = len(X[0])
        for epoch in range(epochs):
            dw = [0.0] * n_features
            db = 0.0
            for i in range(n):
                p = self.predict_proba(X[i])
                error = p - y[i]
                for j in range(n_features):
                    dw[j] += error * X[i][j]
                db += error
            for j in range(n_features):
                self.weights[j] -= self.lr * (dw[j] / n)
            self.bias -= self.lr * (db / n)
            loss = self.compute_loss(X, y)
            self.loss_history.append(loss)
            if epoch % print_every == 0:
                print(f"  Epoch {epoch:4d} | Loss: {loss:.4f} | w: [{self.weights[0]:.3f}, {self.weights[1]:.3f}] | b: {self.bias:.3f}")
        return self

    def accuracy(self, X, y):
        correct = sum(1 for i in range(len(y)) if self.predict(X[i]) == y[i])
        return correct / len(y)




In [14]:
split = int(0.8 * N)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print("\n=== Training Logistic Regression ===")
model = LogisticRegression(n_features=2, learning_rate=0.1)
model.fit(X_train, y_train, epochs=1000, print_every=200)

print(f"\nTrain accuracy: {model.accuracy(X_train, y_train):.4f}")
print(f"Test accuracy:  {model.accuracy(X_test, y_test):.4f}")
print(f"Weights: [{model.weights[0]:.4f}, {model.weights[1]:.4f}]")
print(f"Bias: {model.bias:.4f}")


=== Training Logistic Regression ===
  Epoch    0 | Loss: 0.6289 | w: [0.074, 0.070] | b: -0.001
  Epoch  200 | Loss: 0.3136 | w: [0.464, 0.333] | b: -2.402
  Epoch  400 | Loss: 0.2173 | w: [0.657, 0.497] | b: -3.756
  Epoch  600 | Loss: 0.1723 | w: [0.795, 0.608] | b: -4.684
  Epoch  800 | Loss: 0.1462 | w: [0.902, 0.693] | b: -5.391

Train accuracy: 0.9875
Test accuracy:  1.0000
Weights: [0.9886, 0.7629]
Bias: -5.9623


In [15]:
from sklearn.linear_model import LogisticRegression as SklearnLr
from sklearn.metrics import accuracy_score, precision_score, recall_score,  f1_score
from sklearn.metrics import confusion_matrix , classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

In [17]:
np.random.seed(42)
x_0  = np.random.randn(100,2) + [2,2]
x_1 = np.random.randn(100,2) + [5,5]
x_sk = np.vstack([x_0,x_1])
y_sk =  np.array([0]*100 + [1] *100)

In [18]:
X_tr, X_te, y_tr, y_te = train_test_split(x_sk, y_sk, test_size=0.2, random_state=42)

In [20]:
Scalar = StandardScaler()
x_train_scaled = Scalar.fit_transform(X_tr)
x_test_scaled = Scalar.transform(X_te)

In [21]:
lr =  SklearnLr()
lr.fit(x_train_scaled,y_tr)
y_pred = lr.predict(x_test_scaled)

In [22]:
print("=== Scikit-learn Logistic Regression ===")
print(f"Accuracy:  {accuracy_score(y_te, y_pred):.4f}")
print(f"Precision: {precision_score(y_te, y_pred):.4f}")
print(f"Recall:    {recall_score(y_te, y_pred):.4f}")
print(f"F1:        {f1_score(y_te, y_pred):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_te, y_pred)}")
print(f"\nClassification Report:\n{classification_report(y_te, y_pred)}")

=== Scikit-learn Logistic Regression ===
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1:        1.0000

Confusion Matrix:
[[21  0]
 [ 0 19]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        21
           1       1.00      1.00      1.00        19

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

